# Module 5 lab · Tensors and the linear algebra of models

The code from every lesson of [this module](https://aicourses.dev/research/tensors), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 5.2 Vectors: lists of numbers that point somewhere

A word, a picture or a user can become a list of numbers. Seeing that list as an arrow makes similarity, embeddings and attention things you can picture.

[Read the lesson](https://aicourses.dev/research/tensors/vectors)

### A list is an arrow

In NumPy, a vector is an array:

In [ ]:
import numpy as np

a = np.array([3.0, 4.0])      # 3 steps east, 4 steps north
b = np.array([1.0, -2.0])     # 1 step east, 2 steps south

### Adding and stretching arrows

**Stretching**, or multiplying by a number, multiplies every component. $2\mathbf{a} = (6, 8)$ points the same way and is twice as long. $-1 \times \mathbf{a} = (-3, -4)$ is the same length, pointing exactly backwards. A number used like this, to stretch a vector, is called a **scalar**.

In [ ]:
print(a + b)       # [4. 2.]
print(2 * a)       # [6. 8.]
print(-1 * a)      # [-3. -4.]

### The length of a vector

$$
\lVert \mathbf{a} \rVert = \sqrt{\sum_{i=1}^{n} a_i^2}
$$

In [ ]:
print(np.sqrt(np.sum(a ** 2)))  # 5.0, by Pythagoras
print(np.linalg.norm(a))        # 5.0, built in
print(a / np.linalg.norm(a))    # [0.6 0.8]

### Vectors with hundreds of numbers

You cannot draw an arrow with 768 components, but every rule above works exactly the same way: add component by component, stretch every component, and find the length with the same Σ. NumPy does not care how long the list is:

In [ ]:
rng = np.random.default_rng(seed=0)
v = rng.normal(size=768)    # stand-in for one token's vector
print(v.shape)              # (768,)
print(np.linalg.norm(v))    # 27.689234422059993

## 5.3 Dot products and similarity

A dot product multiplies two lists number by number and adds up the results. It is large when two vectors point the same way. Search uses it to find similar things, and later attention uses it to pick what to look at.

[Read the lesson](https://aicourses.dev/research/tensors/vectors-and-similarity)

### Multiply number by number, then add

You have seen this before: the shopping bill in lesson 3.3, quantities times prices added up, was a dot product. Here are three ways to write it in NumPy, and they all give the same answer:

In [ ]:
import numpy as np

a = np.array([3.0, 4.0])
b = np.array([1.0, 2.0])
print(np.sum(a * b))    # 11.0, multiply then add
print(np.dot(a, b))     # 11.0, built in
print(a @ b)            # 11.0, the @ sign

### Cosine similarity: direction only

It is always between −1 and 1, and stretching either vector by a positive number does not change it. For $(3, 4)$ and $(1, 2)$ the lengths are 5 and $\sqrt{1^2 + 2^2} = \sqrt{5} \approx 2.236$, so the cosine similarity is $\frac{11}{5 \times 2.236} \approx 0.984$: they point almost the same way.

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(cosine(a, b))                    # 0.9838699100999074
print(cosine(a, 10 * a), cosine(a, -a))   # 1.0 -1.0

### Your first search engine

Here is search in miniature. Four made-up films are scored on three made-up qualities, action, romance and comedy, and you describe what you feel like watching as a vector of the same kind. Rank the films by cosine similarity with you:

In [ ]:
titles = ["Robot Rampage", "Paris in Spring",
          "Laugh Riot", "Love and Lasers"]
films = np.array([[0.9, 0.1, 0.2],    # action, romance, comedy
                  [0.1, 0.9, 0.3],
                  [0.2, 0.3, 0.9],
                  [0.7, 0.7, 0.2]])
you = np.array([1.0, 0.2, 0.0])       # mostly action

scores = np.array([cosine(you, f) for f in films])
for i in np.argsort(scores)[::-1]:    # best match first
    print(titles[i], round(scores[i], 3))
# Robot Rampage 0.973
# Love and Lasers 0.816
# Paris in Spring 0.288
# Laugh Riot 0.263

## 5.4 Matrices: a grid of numbers, and a matrix times a vector

A layer of a network is built on a matrix. Multiplying one vector by it, one row at a time, comes before multiplying whole batches at once.

[Read the lesson](https://aicourses.dev/research/tensors/matrices)

### A grid of numbers

This $A$ has shape 2 × 2. The number in row $i$ and column $j$ is written $A_{ij}$, with two subscripts: $A_{12} = 2$ is row 1, column 2. Matrices get capital letters, vectors bold small ones. In NumPy, a matrix is a 2D array, a list of rows:

In [ ]:
import numpy as np

A = np.array([[1.0, 2.0],
              [3.0, 4.0]])
print(A.shape)    # (2, 2)

### A matrix times a vector, row by row

Vectors are written here as a column, top to bottom, which is how they usually appear next to a matrix. The loop says exactly the same thing, and `@` does it in one go:

In [ ]:
x = np.array([5.0, 6.0])
out = np.array([row @ x for row in A])   # one dot product per row
print(out)      # [17. 39.]
print(A @ x)    # [17. 39.]

The **shape rule** follows from the recipe. Each row must have as many numbers as the vector, so a matrix of shape $m \times n$ takes a vector of $n$ numbers and gives back a vector of $m$, one per row. Get it wrong and NumPy refuses:

This cell fails on purpose: the lesson reads the error with you.

In [ ]:
A @ np.array([1.0, 2.0, 3.0])
# ValueError: matmul: Input operand 1 has a mismatch in its core
# dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?)
# (size 3 is different from 2)

### The photo filter, as a matrix

Back to the pixels. Each output colour is one row: how much red, green and blue to mix into it.

In [ ]:
pixel = np.array([200.0, 100.0, 50.0])   # red, green, blue
grey = np.array([[0.299, 0.587, 0.114],
                 [0.299, 0.587, 0.114],
                 [0.299, 0.587, 0.114]])
swap = np.array([[0.0, 0.0, 1.0],      # new red   = old blue
                 [0.0, 1.0, 0.0],      # new green = old green
                 [1.0, 0.0, 0.0]])     # new blue  = old red
print(grey @ pixel)    # [124.2 124.2 124.2]
print(swap @ pixel)    # [ 50. 100. 200.]

## 5.5 Matrix times matrix: many vectors at once

Multiplying a matrix by another matrix is many matrix-times-vector steps done together. Every layer does this to a whole batch of inputs, and it is called matmul for short.

[Read the lesson](https://aicourses.dev/research/tensors/matrix-multiplication)

### Many vectors at once

Here $k$ runs along row $i$ of $A$ and down column $j$ of $B$ at the same time, multiplying the pairs and adding them up, the Σ from lesson 3.3.

In [ ]:
import numpy as np

A = np.array([[1.0, 2.0],
              [3.0, 4.0]])
B = np.array([[5.0, 7.0],
              [6.0, 8.0]])
C = np.zeros((2, 2))
for i in range(2):              # each row of A
    for j in range(2):          # each column of B
        C[i, j] = A[i, :] @ B[:, j]
print(C)        # [[17. 23.]
                #  [39. 53.]]
print(A @ B)    # the same, in one step

### Batches: one example per row

In model code, the batch is usually stored with one example per row: 32 tokens make a 32 × 768 matrix, and four pixels make a 4 × 3 matrix. To apply a filter to every row at once, multiply by the filter’s **transpose**, written $M^\top$ or `M.T`: the same matrix with its rows turned into columns. The black-and-white filter on four pixels:

In [ ]:
grey = np.array([[0.299, 0.587, 0.114],
                 [0.299, 0.587, 0.114],
                 [0.299, 0.587, 0.114]])
pixels = np.array([[200.0, 100.0, 50.0],   # one pixel per row
                   [10.0, 20.0, 30.0],
                   [0.0, 0.0, 255.0],
                   [90.0, 90.0, 90.0]])
out = pixels @ grey.T                      # (4, 3) @ (3, 3)
print(out[:, 0])          # [124.2   18.15  29.07  90.  ]
same = np.array([grey @ p for p in pixels])
print(np.allclose(out, same))              # True

### Order matters

With ordinary numbers, 3 × 4 = 4 × 3. With matrices, the order usually matters:

In [ ]:
print(B @ A)    # [[26. 38.]
                #  [30. 44.]]

## 5.6 Counting the work: how many sums a matmul needs

Each matmul needs a known number of multiplications and additions, called FLOPs. Counting them tells you how long a model takes to run, and in Module 11 how much a training run will cost.

[Read the lesson](https://aicourses.dev/research/tensors/counting-flops)

### Counting a dot product, then a matmul

In code, the formula is one line:

In [ ]:
def matmul_flops(m, n, p):
    return 2 * m * n * p      # m*p entries, each about 2n work

print(matmul_flops(2, 2, 2))          # 16
print(matmul_flops(32, 768, 3072))    # 150994944

### Measuring your runtime

Now time a real one in Colab. Two 1,000 × 1,000 matrices need $2 \times 1000^3 = 2$ billion FLOPs:

In [ ]:
import time
import numpy as np

rng = np.random.default_rng(seed=0)
A = rng.normal(size=(1000, 1000))
B = rng.normal(size=(1000, 1000))
A @ B                                  # warm-up run
start = time.perf_counter()
C = A @ B
seconds = time.perf_counter() - start
print(f"{seconds:.4f} s")
print(f"{matmul_flops(1000, 1000, 1000) / seconds:.3g} FLOP/s")

## 5.7 From NumPy to PyTorch tensors

PyTorch is NumPy that also runs on a GPU and tracks gradients. You meet it on operations you already know before it does anything clever.

[Read the lesson](https://aicourses.dev/research/tensors/pytorch-tensors)

### The same operations, new name

A PyTorch tensor is made just like a NumPy array, and the operations look the same:

In [ ]:
import numpy as np
import torch

a = torch.tensor([3.0, 4.0])
b = torch.tensor([1.0, 2.0])
print(a + b)    # tensor([4., 6.])
print(a @ b)    # tensor(11.)

That is the dot product from lesson 5.3, still 11. A single number comes back as a tensor too, `tensor(11.)`; `.item()` turns it into a plain Python number when you need one. The matmul from lesson 5.5 also matches:

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 7.0],
                  [6.0, 8.0]])
print(A @ B)              # tensor([[17., 23.],
                          #         [39., 53.]])
print(A.shape)            # torch.Size([2, 2])
print(A.mean(dim=0))      # tensor([2., 3.])

### Difference one: 32-bit numbers

In [ ]:
print(a.dtype)                       # torch.float32
print(np.array([3.0, 4.0]).dtype)    # float64

### Difference two: moving between NumPy and PyTorch

You can go both ways:

In [ ]:
n = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(n)    # NumPy array -> tensor
print(t.dtype)             # torch.float64
back = t.numpy()           # tensor -> NumPy array

### Power one: the GPU

Every tensor lives on a **device**: the CPU, or an NVIDIA GPU, which PyTorch calls `"cuda"` and a Colab GPU runtime gives you (lesson 4.5). Moving a tensor is one call, and the arithmetic then runs on that device:

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
At = A.to(device)
print(At.device)          # cpu on Colab’s CPU runtime
print((At @ At).cpu())    # tensor([[ 7., 10.],
                          #         [15., 22.]])

### Power two: slopes for free

In lesson 3.10 you measured the slope of $x^2$ at $x = 3$ by nudging, and the estimates crept towards 6. PyTorch can give you that slope directly:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()       # work out the slope of y with respect to x
print(x.grad)      # tensor(6.)

## 5.8 Tensor shapes: reading and fixing the size of your data

Every tensor has a shape, such as 32 sentences by 10 words by 64 numbers. Many model bugs are shape bugs, so you learn to read and change shapes first.

[Read the lesson](https://aicourses.dev/research/tensors/tensors-and-shapes)

### Reading a shape

A tensor’s shape lists the size of each of its directions, which PyTorch calls **dimensions**, or dims. Each dim usually has a meaning, and the first one is usually the batch.

In [ ]:
import torch

batch = torch.zeros(32, 10, 64)   # sentences, tokens, numbers
print(batch.shape)    # torch.Size([32, 10, 64])
print(batch.ndim)     # 3
print(batch.numel())  # 20480

### Picking parts out

Indexing with a single number removes that dim. A colon, `:`, keeps a dim whole.

In [ ]:
print(batch[0].shape)          # torch.Size([10, 64])
print(batch[:, -1].shape)      # torch.Size([32, 64])
print(batch[:, -1, :5].shape)  # torch.Size([32, 5])

### Reshaping: same numbers, new grid

**`reshape`** pours the same numbers into a new shape. They keep their reading order, left to right and then top to bottom, and the sizes must multiply to the same total.

In [ ]:
x = torch.arange(12)     # 0, 1, ..., 11
m = x.reshape(3, 4)
print(m)
# tensor([[ 0,  1,  2,  3],
#         [ 4,  5,  6,  7],
#         [ 8,  9, 10, 11]])
print(m.reshape(2, -1).shape)    # torch.Size([2, 6])

### Swapping and adding dims

To reorder the dims themselves, use **`permute`**, which takes the old dims in their new order. Images are a common case: you may load a photo as height × width × colour, but PyTorch’s image layers expect colour first.

In [ ]:
img = torch.zeros(64, 64, 3)        # height, width, colour
print(img.permute(2, 0, 1).shape)   # torch.Size([3, 64, 64])

Sometimes a dim of size 1 is needed. Model code expects a batch first, so a single example of 64 numbers must become a batch of one, shape (1, 64). **`unsqueeze`** adds a dim of size 1 where you say, and **`squeeze`** removes it again:

In [ ]:
v = torch.zeros(64)
print(v.unsqueeze(0).shape)              # torch.Size([1, 64])
print(v.unsqueeze(0).squeeze(0).shape)   # torch.Size([64])

## 5.9 Broadcasting and einsum: shorter ways to combine tensors

Broadcasting stretches a small tensor to match a big one without copying it. Einsum writes a combination of tensors as one short line of letters. You will see both often in model code.

[Read the lesson](https://aicourses.dev/research/tensors/broadcasting-and-einsum)

### Broadcasting: stretch the small one

You have already used the simplest case:

In [ ]:
import torch

prices = torch.tensor([3.0, 5.0, 2.0])
print(prices + 1.5)       # tensor([4.5000, 6.5000, 3.5000])

The real use in models is a **bias**: one learned number per feature (per column), added to every example in a batch. It is the $b$ of $wx + b$ from lesson 3.2, not the unfair data of lesson 0.7. The batch has shape (32, 64) and the bias (64,). Broadcasting adds the same 64 numbers to each of the 32 rows:

In [ ]:
batch = torch.zeros(32, 64)
bias = torch.ones(64)
print((batch + bias).shape)        # torch.Size([32, 64])

For (32, 64) and (64,): 64 meets 64, then 32 meets a missing size, which counts as 1 and is stretched. Now try a bias of shape (32,):

This cell fails on purpose: the lesson reads the error with you.

In [ ]:
batch + torch.ones(32)
# RuntimeError: The size of tensor a (64) must match the size
# of tensor b (32) at non-singleton dimension 1

A 3 × 4 times table. From the right, 1 meets 4 (stretch to 4), then 3 meets a missing size (stretch to 3), so the result is (3, 4), and each entry is its row number times its column number:

In [ ]:
rows = torch.arange(1, 4).reshape(3, 1)
cols = torch.arange(1, 5)
print(rows * cols)
# tensor([[ 1,  2,  3,  4],
#         [ 2,  4,  6,  8],
#         [ 3,  6,  9, 12]])

### Broadcasting at work: many cosines at once

In lesson 5.3 you computed cosine similarity in a loop. To do it for whole batches, divide every row by its own length. `norm(dim=1)` gives one length per row, and **`keepdim=True`** keeps the summed dim as size 1, so the lengths come out as a column that broadcasts across each row:

In [ ]:
torch.manual_seed(0)
x = torch.randn(5, 3)                    # 5 vectors of 3 numbers
lengths = x.norm(dim=1, keepdim=True)    # shape (5, 1)
unit = x / lengths                       # every row now length 1
print(unit.norm(dim=1))   # tensor([1.0000, 1.0000, ...])

### Einsum: name the dims, then say what to keep

Here are this module’s operations, in einsum:

In [ ]:
a = torch.tensor([3.0, 4.0])
b = torch.tensor([1.0, 2.0])
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 7.0], [6.0, 8.0]])

print(torch.einsum("i,i->", a, b))       # tensor(11.)
print(torch.einsum("ij,j->i", A, b))     # tensor([ 5., 11.])
print(torch.einsum("ik,kj->ij", A, B))   # [[17, 23], [39, 53]]
print(torch.einsum("ij->ji", A))         # the transpose

The payoff comes with more dims. Module 10’s attention compares every query token with every key token, inside every item of a batch. A token’s query says what it is looking for and its key says what it offers; lesson 10.1 explains both. With b for batch, q and k for the two kinds of token and d for the numbers in each:

In [ ]:
q = torch.randn(2, 4, 8)      # batch, query tokens, numbers
k = torch.randn(2, 6, 8)      # batch, key tokens, numbers
scores = torch.einsum("bqd,bkd->bqk", q, k)
print(scores.shape)           # torch.Size([2, 4, 6])

## 5.10 Project: vectorised semantic search

Puts vectors, matmul costs and shapes to work on a real search you can query.

[Read the lesson](https://aicourses.dev/research/tensors/project-semantic-search)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Steps

1. **Get the sentences.** In a new Colab notebook, download the plain-text book from Project Gutenberg into the runtime as `alice.txt`, then cut it into sentences:

In [ ]:
import re, urllib.request

url = "https://www.gutenberg.org/ebooks/11.txt.utf-8"
urllib.request.urlretrieve(url, "alice.txt")       # save a copy
text = open("alice.txt", encoding="utf-8").read()
start = text.find("\n", text.find("*** START"))  # end of that line
end = text.find("*** END")
body = " ".join(text[start:end].split())         # tidy the spaces
parts = re.split(r"(?<=[.!?])\s+", body)         # cut after . ! ?
sentences = [s for s in parts if len(s.split()) >= 4]
print(len(sentences))                            # 926

2. **Embed them.** The encoder gives one vector per token; average them into one per sentence, ignoring the padding tokens added to make sentences the same length. Broadcasting does it without a loop:

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

name = "sentence-transformers/all-MiniLM-L6-v2"
tok = AutoTokenizer.from_pretrained(name)
enc = AutoModel.from_pretrained(name)

def mean_pool(hidden, mask):
    mask = mask[..., None].float()     # (batch, tokens, 1)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1)

def embed(texts):
    batch = tok(texts, padding=True, truncation=True,
                return_tensors="pt")
    with torch.no_grad():
        hidden = enc(**batch).last_hidden_state
    return mean_pool(hidden, batch["attention_mask"])

3. **Search without loops.** Normalise every row (divide it by its length, so it has length 1) with `norm(dim=1, keepdim=True)`, as in lesson 5.9, embed the query the same way, and compute every similarity with one matmul. `topk(5, dim=1)` returns the five best scores and their positions for each query. Your function must also accept several queries at once, giving a (queries × 926) table.
4. **Count the FLOPs.** Write the formula, 2 × queries × 384 × 926, then check it with PyTorch’s own counter:

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

with FlopCounterMode(display=False) as counter:
    sims = q @ d.T       # q: (3, 384), d: (926, 384), normalised
print(counter.get_total_flops(), 2 * 3 * 384 * 926)

### The shape kata

A test looks like this, for puzzle 1:

In [ ]:
x = torch.randn(32, 10, 64)
out = x[:, -1]
assert out.shape == (32, 64)
assert torch.equal(out[0], x[0, 9])   # the last of 10 tokens